# Compress the Complete `op/` Folder

This notebook compresses the entire project `op/` output folder into a timestamped `.zip` archive.

It is designed to be placed at:

```text
notebooks/misc_op.ipynb
```

Run all cells from the project environment. The notebook automatically searches for the project root by locating an `op/` folder in the current working directory or parent directories.

In [1]:
from pathlib import Path
from datetime import datetime
import zipfile
import os


def find_project_root(start: Path | None = None, required_dir: str = "op") -> Path:
    """Find the nearest parent directory containing the required output folder."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / required_dir).is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find a project root containing an '{required_dir}/' folder. "
        f"Current working directory: {start}"
    )


PROJECT_ROOT = find_project_root()
OP_DIR = PROJECT_ROOT / "op"
ARCHIVE_DIR = PROJECT_ROOT / "arch"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_PATH = ARCHIVE_DIR / f"op_full_archive_{timestamp}.zip"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input folder : {OP_DIR}")
print(f"Output ZIP   : {ZIP_PATH}")

Project root: /home/ppanta/puru_proj/1_dementia_progress/dementia_progress_v0
Input folder : /home/ppanta/puru_proj/1_dementia_progress/dementia_progress_v0/op
Output ZIP   : /home/ppanta/puru_proj/1_dementia_progress/dementia_progress_v0/arch/op_full_archive_20260529_132056.zip


In [2]:
def format_bytes(num_bytes: int) -> str:
    """Human-readable file size."""
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024


def collect_files(folder: Path) -> list[Path]:
    """Collect all regular files under a folder."""
    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder}")
    if not folder.is_dir():
        raise NotADirectoryError(f"Expected a directory, got: {folder}")
    return sorted(p for p in folder.rglob("*") if p.is_file())


files = collect_files(OP_DIR)
total_bytes = sum(p.stat().st_size for p in files)

print(f"Files found      : {len(files):,}")
print(f"Uncompressed size: {format_bytes(total_bytes)}")

if not files:
    raise RuntimeError(f"No files were found under {OP_DIR}")

Files found      : 78
Uncompressed size: 246.51 MB


In [3]:
def zip_folder(source_folder: Path, zip_path: Path, project_root: Path) -> Path:
    """
    Compress a folder into a ZIP archive.

    The archive preserves paths relative to the project root, so files appear as:
    op/...
    """
    source_folder = source_folder.resolve()
    zip_path = zip_path.resolve()
    project_root = project_root.resolve()

    files_to_zip = collect_files(source_folder)

    try:
        from tqdm.auto import tqdm
        iterator = tqdm(files_to_zip, desc="Compressing op folder", unit="file")
    except Exception:
        iterator = files_to_zip

    with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for file_path in iterator:
            arcname = file_path.relative_to(project_root)
            zf.write(file_path, arcname=arcname)

    return zip_path


created_zip = zip_folder(OP_DIR, ZIP_PATH, PROJECT_ROOT)
zip_size = created_zip.stat().st_size

print("Compression complete.")
print(f"Created ZIP: {created_zip}")
print(f"ZIP size   : {format_bytes(zip_size)}")

/home/ppanta/puru_proj/1_dementia_progress/dementia_progress_v0/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Compressing op folder: 100%|██████████████████| 78/78 [00:07<00:00, 10.68file/s]

Compression complete.
Created ZIP: /home/ppanta/puru_proj/1_dementia_progress/dementia_progress_v0/arch/op_full_archive_20260529_132056.zip
ZIP size   : 40.32 MB


In [4]:
def validate_zip(zip_path: Path) -> None:
    """Validate ZIP integrity and show a small preview of archived files."""
    with zipfile.ZipFile(zip_path, mode="r") as zf:
        bad_file = zf.testzip()
        if bad_file is not None:
            raise RuntimeError(f"ZIP validation failed at: {bad_file}")

        names = zf.namelist()
        print(f"ZIP integrity: passed")
        print(f"Archived files: {len(names):,}")
        print("\nFirst 20 archived paths:")
        for name in names[:20]:
            print(f"  {name}")


validate_zip(created_zip)

ZIP integrity: passed
Archived files: 78

First 20 archived paths:
  op/dementia_progression_20260527_190631/.ipynb_checkpoints/manuscript_readiness_gate_report-checkpoint.csv
  op/dementia_progression_20260527_190631/.ipynb_checkpoints/manuscript_readiness_overall_status-checkpoint.csv
  op/dementia_progression_20260527_190631/logs/config.log
  op/dementia_progression_20260527_190631/logs/gpu.log
  op/dementia_progression_20260527_190631/logs/launcher.log
  op/dementia_progression_20260527_190631/logs/llm.log
  op/dementia_progression_20260527_190631/logs/ollama_generate_probe_response.json
  op/dementia_progression_20260527_190631/logs/ollama_server.log
  op/dementia_progression_20260527_190631/logs/pipeline.log
  op/dementia_progression_20260527_190631/logs/runtime.log
  op/dementia_progression_20260527_190631/logs/warnings.log
  op/dementia_progression_20260527_190631/manuscript_readiness_gate_report.csv
  op/dementia_progression_20260527_190631/manuscript_readiness_overall_status.

## Output

The compressed file is saved under:

```text
misc/op_full_archive_YYYYMMDD_HHMMSS.zip
```

The archive keeps the original `op/` directory structure inside the ZIP.